# Strands GraphRAG Agent

In the previous notebooks you built two retrieval strategies — `VectorRetriever` for pure semantic search and `VectorCypherRetriever` for graph-enriched search. In Lab 3 you built a Strands agent with simple tools. This notebook combines those ideas: a Strands agent that has both retrievers as tools and **decides which one to use** based on the question.

**Learning Objectives:**
- Wrap neo4j-graphrag retrievers as Strands `@tool` functions
- Build an agent that chooses between retrieval strategies
- Inspect the agent's tool selection reasoning

The key shift is **who picks the retrieval strategy**. In notebooks 02 and 03, the developer hardcoded which retriever to use. Here, the model decides.

In [ ]:
%pip install strands-agents "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

## Setup

Import Strands alongside the neo4j-graphrag components from the previous notebooks. The configuration pattern is the same — credentials from `CONFIG.txt`, a Neo4j driver, and the Bedrock embedder for vector search. The LLM is handled by Strands (`BedrockModel`) rather than the neo4j-graphrag `BedrockLLM`, since the agent manages the conversation loop directly.

In [ ]:
import os

import neo4j
from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from strands import Agent, tool
from strands.models import BedrockModel

from lib.data_utils import get_embedder

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()

print(f'Connected to Neo4j!')
print(f'Embedder: {embedder.model_id}')

## Initialize Retrievers

Set up both retrievers from the previous notebooks. The `VectorRetriever` returns matched chunks only; the `VectorCypherRetriever` adds graph context — companies, products, and risk factors linked to each chunk.

In [ ]:
# Vector retriever (from notebook 02)
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

# Retrieval query (from notebook 03)
RETRIEVAL_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
WITH node, doc, score, company
RETURN node.text AS text,
       score,
       {document: doc.accessionNumber,
        filingType: doc.filingType,
        company: company.name,
        products: collect { MATCH (p:Product)-[:FROM_CHUNK]->(node) RETURN p.name },
        risks: collect { MATCH (r:RiskFactor)-[:FROM_CHUNK]->(node) RETURN r.name }
       } AS metadata
"""

def format_record(record: neo4j.Record) -> RetrieverResultItem:
    """Separate chunk text (content for LLM) from structured graph metadata."""
    metadata = record.get("metadata") or {}
    metadata["score"] = record.get("score")
    return RetrieverResultItem(
        content=record.get("text", ""),
        metadata=metadata,
    )

# VectorCypher retriever (from notebook 03)
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=RETRIEVAL_QUERY,
    result_formatter=format_record,
)

print('Both retrievers initialized!')

## Wrap Retrievers as Agent Tools

The `@tool` decorator turns any function into something a Strands agent can call. The docstring is critical — it's what the model reads to decide *when* to use each tool. Notice how the docstrings explain when each tool is appropriate, not just what it does.

Each tool runs a retriever and formats the results as a string that the agent can reason over.

In [ ]:
@tool
def semantic_search(query: str, top_k: int = 5) -> str:
    """Search SEC 10-K filing chunks by semantic similarity.

    Use this for broad or thematic questions where the text content
    of the filing chunks is sufficient to answer — for example,
    summarizing key themes, finding specific passages, or answering
    questions that don't require knowing which company or product
    is involved.

    Args:
        query: The search query.
        top_k: Number of chunks to return (default 5).

    Returns:
        The matching chunks with similarity scores.
    """
    result = vector_retriever.search(query_text=query, top_k=top_k)
    chunks = []
    for item in result.items:
        score = item.metadata.get('score', 0.0)
        chunks.append(f"[Score: {score:.4f}] {item.content}")
    return "\n\n".join(chunks)


@tool
def graph_enriched_search(query: str, top_k: int = 5) -> str:
    """Search SEC 10-K filing chunks with graph-enriched context.

    Use this when the question involves specific companies, products,
    or risk factors. The graph traversal adds structured entity
    information to each chunk — company names, products they offer,
    and risk factors they face — so you can answer entity-specific
    questions with precision.

    Args:
        query: The search query.
        top_k: Number of chunks to return (default 5).

    Returns:
        The matching chunks with similarity scores and entity metadata.
    """
    result = vector_cypher_retriever.search(query_text=query, top_k=top_k)
    chunks = []
    for item in result.items:
        meta = item.metadata or {}
        header = (f"[Score: {meta.get('score', 0):.4f}] "
                  f"Company: {meta.get('company', 'N/A')} | "
                  f"Products: {meta.get('products', [])} | "
                  f"Risks: {meta.get('risks', [])}")
        chunks.append(f"{header}\n{item.content}")
    return "\n\n".join(chunks)


print('Tools defined: semantic_search, graph_enriched_search')

## Create the Agent

The agent gets both tools and a system prompt that describes the knowledge graph. Strands handles the ReAct loop internally — the model reasons about which tool fits each question, calls it, observes the results, and synthesizes an answer.

This is the same `Agent` + `BedrockModel` pattern from Lab 3, but now the tools do something meaningful: they search a knowledge graph.

In [ ]:
MODEL_ID = os.getenv('MODEL_ID', 'us.anthropic.claude-sonnet-4-5-20250929-v1:0')
REGION = os.getenv('REGION', 'us-east-1')

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

agent = Agent(
    model=bedrock_model,
    system_prompt="""You are a financial research assistant with access to SEC 10-K filings stored in a Neo4j knowledge graph.

You have two search tools:
- semantic_search: finds relevant text chunks by meaning — use for broad or thematic questions
- graph_enriched_search: finds chunks AND returns connected entities (companies, products, risk factors) — use when the question involves specific companies or relationships

Choose the tool that best fits each question. Always ground your answers in the retrieved data.""",
    tools=[semantic_search, graph_enriched_search],
)

print(f'Agent created with model: {MODEL_ID}')

## Test the Agent

Try questions that should trigger different tool choices. The agent streams its reasoning to the output — watch for which tool it selects and why.

### General question (should pick `semantic_search`)

In [ ]:
response = agent("Summarize the key themes across the 10-K filings.")

### Entity-specific question (should pick `graph_enriched_search`)

In [ ]:
response = agent("What products does Apple offer and what risks do they face?")

### Multi-entity comparison (may use both tools or multiple calls)

In [ ]:
response = agent("Compare the risk factors between Apple and Microsoft. Which company faces more technology-related risks?")

## Inspect Tool Usage

After each invocation, the agent's conversation history contains every message in the ReAct loop: the model's reasoning, tool calls it made, and tool results it received. Walk through the messages to see exactly which tools the agent chose and what data it got back.

In [ ]:
for msg in agent.messages:
    role = msg['role']
    for content in msg.get('content', []):
        if 'toolUse' in content:
            tool_use = content['toolUse']
            print(f"[{role}] Called tool: {tool_use['name']}")
            print(f"         Input: {tool_use['input']}")
            print()
        elif 'toolResult' in content:
            tool_result = content['toolResult']
            status = tool_result.get('status', 'unknown')
            # Show a preview of the result content
            result_text = ''
            for block in tool_result.get('content', []):
                if 'text' in block:
                    result_text = block['text'][:200]
                    break
            print(f"[{role}] Tool result ({status}): {result_text}...")
            print()

## Try Your Own Questions

Experiment with different questions. Think about what kind of question would lead the agent to choose one tool over the other — or to call both.

In [ ]:
response = agent("YOUR QUESTION HERE")

## Summary

This notebook combined the retrievers from notebooks 02 and 03 with the Strands agent pattern from Lab 3. The key progression across the workshop:

| Notebook | Pattern | Who decides what to retrieve? |
|----------|---------|-------------------------------|
| 02 — VectorRetriever | `GraphRAG` pipeline | Developer (hardcoded) |
| 03 — VectorCypherRetriever | `GraphRAG` pipeline | Developer (hardcoded) |
| **04 — Strands Agent** | **Agent with both retrievers as tools** | **The model** |
| 05 (next) — MCP Agent | Agent with MCP tools | The model, via MCP transport |

The shift from 03 to 04 is **who picks the retrieval strategy** — the model selects the right tool based on the question. The shift from 04 to 05 is **how the tools are delivered**: local `@tool` functions here become remote tools served over MCP in Lab 5.

---

**Next:** [Lab 5: Neo4j MCP Server](../Lab_5_MCP_Server/) — agent-based retrieval via MCP

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')